# 01 · Treino do classificador de maturação do tomate (YOLO26n-cls)

Treina o modelo que o app usa para classificar a foto em **verde**, **de vez**, **maduro**, **passado** ou **outro**, a partir do dataset público [Unified Tomato Detection & Ripeness Dataset](https://huggingface.co/datasets/ParsaVictor/tomato-ripeness-detection) (CC BY 4.0).

**Como rodar:** Ambiente de execução → Alterar o tipo de ambiente → **GPU T4**. Depois, Executar tudo. Os resultados (pesos, métricas, matriz de confusão e modelo exportado) vão para `MyDrive/tcc-maturacao/` no Google Drive.

**Integridade:** os números que entram na monografia saem das execuções deste notebook e ficam salvos em `metricas_teste.json` e `ambiente.json`. Nada de copiar número de outro lugar.

## 1. Ambiente
Versão do `ultralytics` fixada para o experimento ser reprodutível (registrar na Metodologia).

In [ ]:
!pip install -q ultralytics==8.4.188 huggingface_hub
import platform, torch, ultralytics
from ultralytics import YOLO
ultralytics.checks()
print('torch', torch.__version__, '| CUDA disponível:', torch.cuda.is_available())

## 2. Configuração
O mapeamento de classes segue as decisões do projeto (ver `CLAUDE.md`).

In [ ]:
from pathlib import Path

SEED = 42
DATASET_REPO = 'ParsaVictor/tomato-ripeness-detection'
DATASET_REV = '91481266b3674f4eec31e72310888321d0e102f2'  # commit do dataset fixado (versão de 27/10/2026)
NOME_EXPERIMENTO = 'yolo26n_cls_v1'
GERAR_OUTRO = True  # cria a classe "outro" com recortes de fundo, sem tomate (seção 5)
LIMITE_OUTRO = {'train': 4000, 'val': 700, 'test': 150}  # recortes "outro" por split

# Classes do dataset -> classes do app
MAPA_CLASSES = {'raw': 'verde', 'half_ripe': 'de_vez', 'ripe': 'maduro', 'rotten': 'passado'}

from google.colab import drive
drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive/tcc-maturacao')
RUNS = DRIVE / 'runs'
EXPORTS = DRIVE / 'exports'
for pasta in (RUNS, EXPORTS):
    pasta.mkdir(parents=True, exist_ok=True)

DATA = Path('/content/data')  # disco local do Colab: bem mais rápido que o Drive para treinar
DATA.mkdir(exist_ok=True)

## 3. Download dos recortes de classificação
O arquivo `tomato_ripeness_crops.zip` já vem separado em `train`, `val` e `test`, com recortes de 160×160 de um tomate cada.

In [ ]:
import zipfile
from huggingface_hub import hf_hub_download

zip_recortes = hf_hub_download(DATASET_REPO, 'tomato_ripeness_crops.zip', repo_type='dataset', revision=DATASET_REV)
with zipfile.ZipFile(zip_recortes) as z:
    z.extractall(DATA / 'crops_raw')
origem = next((DATA / 'crops_raw').rglob('cls_tomato'))
print('pasta de origem:', origem)

## 4. Classes do app
Copia cada classe do dataset para a pasta com o nome usado no app.

In [ ]:
import shutil

DATASET = DATA / 'tomate_cls'
if DATASET.exists():
    shutil.rmtree(DATASET)
for split in ('train', 'val', 'test'):
    for original, nova in MAPA_CLASSES.items():
        shutil.copytree(origem / split / original, DATASET / split / nova)

def contar(raiz):
    for split in ('train', 'val', 'test'):
        pastas = sorted(p for p in (raiz / split).iterdir() if p.is_dir())
        print(split, {p.name: sum(1 for _ in p.iterdir()) for p in pastas})

contar(DATASET)

**Atenção ao interpretar os números:**
- A classe `passado` (rotten) foi multiplicada por 4 no `train` pelo autor do dataset, com espelhamentos e rotações.
- O autor mapeou rótulos como `Fresh` para `ripe`. Pode haver tomate "fresco", mas não maduro, dentro de `maduro`; olhe a amostra abaixo.

In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import Image

random.seed(SEED)
classes = sorted(p.name for p in (DATASET / 'train').iterdir())
fig, eixos = plt.subplots(len(classes), 6, figsize=(12, 2.2 * len(classes)))
for i, classe in enumerate(classes):
    amostra = random.sample(sorted((DATASET / 'train' / classe).iterdir()), 6)
    for j, arquivo in enumerate(amostra):
        eixos[i][j].imshow(Image.open(arquivo))
        eixos[i][j].axis('off')
    eixos[i][0].set_title(classe, loc='left')
plt.tight_layout()
plt.show()

## 5. Classe "outro" (opcional, recomendado)
Recortes de fundo (folhas, esteira, caixas, mesa) tirados das imagens de detecção do mesmo dataset, **sem sobrepor nenhuma caixa de tomate**. Ensinam o modelo a responder "isto não é um tomate" em vez de forçar uma das quatro classes. Cada split usa as imagens do split correspondente, para não vazar informação entre treino e teste.

In [ ]:
import numpy as np

PASTAS_DETECCAO = {'train': 'train', 'val': 'valid', 'test': 'test'}

def caixas_em_pixels(label_path, largura, altura):
    caixas = []
    if label_path.exists():
        for linha in label_path.read_text().splitlines():
            partes = linha.split()
            if len(partes) >= 5:
                _, xc, yc, bw, bh = map(float, partes[:5])
                caixas.append(((xc - bw / 2) * largura, (yc - bh / 2) * altura,
                               (xc + bw / 2) * largura, (yc + bh / 2) * altura))
    return caixas

def sobrepoe(a, b):
    return not (a[2] <= b[0] or b[2] <= a[0] or a[3] <= b[1] or b[3] <= a[1])

if GERAR_OUTRO:
    zip_deteccao = hf_hub_download(DATASET_REPO, 'tomato_unified_detection.zip', repo_type='dataset', revision=DATASET_REV)
    with zipfile.ZipFile(zip_deteccao) as z:
        z.extractall(DATA / 'det_raw')
    unified = next((DATA / 'det_raw').rglob('unified'))
    rng = np.random.default_rng(SEED)
    for split, pasta in PASTAS_DETECCAO.items():
        imagens = sorted(p for p in unified.glob(f'*/{pasta}/images/*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
        rng.shuffle(imagens)
        destino = DATASET / split / 'outro'
        destino.mkdir(parents=True, exist_ok=True)
        feitos = 0
        for img_path in imagens:
            if feitos >= LIMITE_OUTRO[split]:
                break
            label_path = img_path.parent.parent / 'labels' / (img_path.stem + '.txt')
            with Image.open(img_path) as im:
                im = im.convert('RGB')
                largura, altura = im.size
                caixas = caixas_em_pixels(label_path, largura, altura)
                lado = int(min(largura, altura) * rng.uniform(0.15, 0.35))
                for _ in range(20):  # tentativas de achar um recorte sem tomate
                    x0 = int(rng.integers(0, largura - lado))
                    y0 = int(rng.integers(0, altura - lado))
                    recorte = (x0, y0, x0 + lado, y0 + lado)
                    if not any(sobrepoe(recorte, c) for c in caixas):
                        im.crop(recorte).resize((160, 160)).save(destino / f'outro_{split}_{feitos:05d}.jpg', quality=92)
                        feitos += 1
                        break
        print(f'{split}: {feitos} recortes "outro" de {len(imagens)} imagens disponíveis')

contar(DATASET)

## 6. Treino
Transfer learning a partir dos pesos `yolo26n-cls.pt` (pré-treinados no ImageNet). Os checkpoints vão para o Drive a cada 5 épocas.

In [ ]:
pasta_exp = RUNS / NOME_EXPERIMENTO
# Trava: rodar tudo de novo criaria outra pasta de treino, e as seções seguintes leriam a antiga.
if (pasta_exp / 'weights' / 'last.pt').exists():
    raise RuntimeError(
        f'Já existe um treino em {pasta_exp}. Para continuar esse treino, use a célula de retomada, logo abaixo. '
        'Para treinar de novo do zero, mude NOME_EXPERIMENTO na seção 2 (por exemplo, para yolo26n_cls_v2).'
    )

modelo = YOLO('yolo26n-cls.pt')
resultados = modelo.train(
    data=str(DATASET),
    epochs=50,
    imgsz=224,
    batch=64,
    patience=15,       # para cedo se a validação parar de melhorar
    seed=SEED,
    deterministic=True,
    project=str(RUNS),
    name=NOME_EXPERIMENTO,
    exist_ok=True,     # grava sempre em RUNS/NOME_EXPERIMENTO, a pasta que as próximas seções leem
    save_period=5,
)

**Se o Colab cair no meio do treino**, a sessão recomeça do zero: o disco local é apagado e o pacote precisa ser instalado de novo. Rode as seções 1 a 5, pule a célula de treino acima e rode a célula abaixo, sem o `#`. Ela retoma do último checkpoint salvo no Drive. Depois, siga da seção 7 em diante.

Se rodar tudo de novo por engano, a célula de treino para com um aviso e não apaga nada.

In [ ]:
# YOLO(str(RUNS / NOME_EXPERIMENTO / 'weights' / 'last.pt')).train(resume=True)

## 7. Avaliação no conjunto de teste do dataset
Mede acurácia, precisão, recall e F1 por classe e gera a matriz de confusão. **Isto avalia recortes do dataset público** (muitas fotos de esteira e estufa), da mesma origem do treino, e por isso tende a ser otimista em relação a fotos de campo. A avaliação com fotos próprias ficou como trabalho futuro (decisão de 16/11/2026).

In [ ]:
import json
import time
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix

pasta_exp = RUNS / NOME_EXPERIMENTO
modelo = YOLO(str(pasta_exp / 'weights' / 'best.pt'))
metricas = modelo.val(data=str(DATASET), split='test', imgsz=224)
print('top-1 no teste:', metricas.top1)

nomes = modelo.names
y_true, y_pred, tempos = [], [], []
for pasta in sorted(p for p in (DATASET / 'test').iterdir() if p.is_dir()):
    for arquivo in sorted(pasta.iterdir()):
        inicio = time.perf_counter()
        r = modelo.predict(str(arquivo), imgsz=224, verbose=False)[0]
        tempos.append(time.perf_counter() - inicio)
        y_true.append(pasta.name)
        y_pred.append(nomes[int(r.probs.top1)])

rotulos = sorted(set(y_true))
print(classification_report(y_true, y_pred, labels=rotulos, digits=3))

matriz = confusion_matrix(y_true, y_pred, labels=rotulos)
ConfusionMatrixDisplay(matriz, display_labels=rotulos).plot(cmap='Greens', xticks_rotation=45)
plt.title('Matriz de confusão: teste do dataset público')
plt.tight_layout()
plt.savefig(pasta_exp / 'matriz_confusao_teste.png', dpi=200)
plt.show()

relatorio = classification_report(y_true, y_pred, labels=rotulos, output_dict=True)
relatorio['top1_ultralytics'] = float(metricas.top1)
relatorio['tempo_medio_por_imagem_s_colab'] = sum(tempos) / len(tempos)  # GPU do Colab, não o celular
(pasta_exp / 'metricas_teste.json').write_text(json.dumps(relatorio, indent=2, ensure_ascii=False))
print('salvo em', pasta_exp / 'metricas_teste.json')

## 8. Registro do ambiente
Tudo o que a Metodologia precisa para o experimento ser reproduzido. Fica antes da exportação para ser gravado ainda na sessão que treinou o modelo.

In [ ]:
import json

pasta_exp = RUNS / NOME_EXPERIMENTO
registro = {
    'experimento': NOME_EXPERIMENTO,
    'ultralytics': ultralytics.__version__,
    'torch': torch.__version__,
    'python': platform.python_version(),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'dataset': DATASET_REPO,
    'dataset_revision': DATASET_REV,
    'mapa_classes': MAPA_CLASSES,
    'classe_outro': LIMITE_OUTRO if GERAR_OUTRO else None,
    'seed': SEED,
    'treino': {'modelo_base': 'yolo26n-cls.pt', 'epochs': 50, 'imgsz': 224, 'batch': 64, 'patience': 15},
}
(pasta_exp / 'ambiente.json').write_text(json.dumps(registro, indent=2, ensure_ascii=False))
print(json.dumps(registro, indent=2, ensure_ascii=False))

## 9. Exportação para o celular
LiteRT (`.tflite`) para Android e Core ML para iOS, com o mesmo tamanho de imagem do treino. O Core ML às vezes falha no Linux do Colab; se falhar, exporte depois num Mac com o mesmo `best.pt`.

Se a exportação falhar ou a sessão cair aqui, o treino e as métricas já estão no Drive: rode as seções 1 e 2 e depois a célula abaixo.

In [ ]:
import shutil

modelo = YOLO(str(RUNS / NOME_EXPERIMENTO / 'weights' / 'best.pt'))  # melhor época do treino, salva no Drive
exportados = []
arquivo_tflite = modelo.export(format='litert', imgsz=224)  # nas versões antigas do pacote o nome era 'tflite'
exportados.append(Path(arquivo_tflite))
try:
    exportados.append(Path(modelo.export(format='coreml', imgsz=224)))
except Exception as erro:
    print('Core ML não exportou aqui; fazer num Mac:', erro)

for caminho in exportados:
    destino = EXPORTS / NOME_EXPERIMENTO / caminho.name
    destino.parent.mkdir(parents=True, exist_ok=True)
    if caminho.is_dir():
        shutil.copytree(caminho, destino, dirs_exist_ok=True)
    else:
        shutil.copy2(caminho, destino)
    print('copiado para', destino)

## Próximos passos
1. Copiar o `.tflite` de `MyDrive/tcc-maturacao/exports/` para `app/assets/models/` no repositório.
2. Levar `metricas_teste.json`, `matriz_confusao_teste.png` e `ambiente.json` para a monografia: Metodologia (treinamento) e Resultados.
3. Conferir, com as mesmas imagens, se o app dá a mesma classe que o Python. A avaliação com fotos próprias ficou como trabalho futuro (decisão de 16/11/2026).